# 12 — The level: cohort drift and its repair

Notebook 08 found that completion within 5 years fell from 12.8% for 2017 entrants to 8.3% and 9.6% for 2018 and 2019 entrants, while LightGBM, fitted on entrants to 2016, stayed at 13–14%: a tree cannot extrapolate the request year. For a portfolio the level matters as much as the ranking, because expected MW online scales with it. This notebook asks what drifted (the hazard of operation, of withdrawal, or both), repairs the level of the v2 model with a rolling recalibration that never sees the cohort it scores, reports slope **and** intercept on the 2018–2019 entrants (criterion C7), and holds the models' first- and second-year predictions against the 2020–2022 entrants' outcomes from both editions of the LBNL file.

**Inputs → Outputs**

| | Path | Notes |
|---|---|---|
| in | `data/interim/trackA.parquet`, `data/interim/queue_clean.parquet`, `results/dt_tuning.json` | notebooks 02/04/06 |
| in | `data/processed/predictions_v2.parquet`, `results/variant_chosen.json` | notebook 11 — the v2 model and both baselines on the test cohort |
| in | `results/early_incidence_2020_2022.csv` | notebook 01b — observed 1- and 2-year incidence of the 2020–2022 entrants, both editions |
| in | `results/success_criteria_v2_registered.json` | notebook 11 — bootstrap settings |
| out | `results/drift_by_cohort.csv`, `results/drift_by_cohort_region.csv`, `results/drift_hazards_by_cohort.csv` | the diagnosis |
| out | `results/recalibration.csv` | the rolling intercepts per scored year and region |
| out | `results/level_repair_metrics.csv` | slope and intercept with intervals, per option and cohort (C7's inputs) |
| out | `results/early_hazards_2020_2022.csv` | predicted against observed P(operation by 1 and 2 years), 2020–2022 entrants |
| out | `data/processed/predictions_v2_recalibrated.parquet` | the v2 model's raw and recalibrated test predictions |
| out | `figures/F11_level_by_entry_year.*`, `figures/12_level_repair/*` | F11 and the notebook's figures |
| out | `results/level_checks.csv` | this notebook's checks |

## Setup

In [ ]:
import json
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().resolve().parent))  # notebooks run from notebooks/; src/ is one level up
from src import config, discrete_time, drift, evaluation, paths, plots, survival_tools, variants

paths.ensure_dirs()
plots.set_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)

NOTEBOOK = "12_level_repair"
TRACK_A = config.TRACK_A_REGIONS
START = time.time()
REGISTERED = json.loads((paths.RESULTS / "success_criteria_v2_registered.json").read_text(encoding="utf-8"))
N_BOOT, LEVEL, SEED = REGISTERED["n_boot"], REGISTERED["level"], REGISTERED["seed_v2"] + 100
LEVEL_HORIZONS = [3, 5]  # the horizons C7 names; 7 years is observed for 2017 entrants only
SOURCE_YEAR = {2017: 2016, 2018: 2017, 2019: 2018}  # each scored cohort takes its level from the cohort before it
TEST_YEARS = list(SOURCE_YEAR)


def save(fig, name, caption):
    """Save a figure of this notebook under figures/12_level_repair/ with its caption stub."""
    return plots.save_fig(fig, name, caption, NOTEBOOK, folder=NOTEBOOK)


def pct(x, digits=1):
    return f"{100 * x:.{digits}f}%"

## 1. Data and the v2 model

The v2 model is the candidate notebook 11 chose on 2016 entrants (`results/variant_chosen.json`), refitted here twice: on entrants to 2015, to predict the 2016 cohort out of sample (the level source for 2017), and on entrants to 2016, to predict 2017–2019 as notebook 11 did (the refit is checked against notebook 11's stored predictions).

In [ ]:
clean = pd.read_parquet(paths.DATA_INTERIM / "queue_clean.parquet")
track_a = pd.read_parquet(paths.DATA_INTERIM / "trackA.parquet")
track_a["tech"] = track_a["type_clean"].map(plots.tech_group)
tuning = json.loads((paths.RESULTS / "dt_tuning.json").read_text(encoding="utf-8"))
CHOSEN = json.loads((paths.RESULTS / "variant_chosen.json").read_text(encoding="utf-8"))["chosen"]
SPEC = variants.CANDIDATES[CHOSEN]
pred_v2 = pd.read_parquet(paths.DATA_PROCESSED / "predictions_v2.parquet")

tune = track_a[track_a["q_year"] <= config.TUNE_MAX_YEAR]
final = track_a[track_a["q_year"] <= config.TRAIN_MAX_YEAR]
cohort_2016 = track_a[track_a["q_year"] == config.VALIDATION_YEAR].copy()
test = track_a[track_a["q_year"].between(*config.TEST_YEARS)].copy()
test["large"] = variants.at_least_20(test)
cohort_2016["large"] = variants.at_least_20(cohort_2016)

fit_tune = variants.fit_candidate(SPEC, tune, clean, tuning)
fit_final = variants.fit_candidate(SPEC, final, clean, tuning)
p_2016 = discrete_time.predict_both(fit_tune, cohort_2016)[SPEC["model"]]
p_test = discrete_time.predict_both(fit_final, test)[SPEC["model"]]
stored = {h: pred_v2[pred_v2["horizon"] == h].set_index("project_key").loc[test["project_key"].astype(str), "chosen"].to_numpy()
          for h in config.HORIZONS}
refit_gap = max(float(np.abs(p_test[h].to_numpy() - stored[h]).max()) for h in config.HORIZONS)
assert refit_gap < 1e-6, f"the refit differs from notebook 11's stored predictions by {refit_gap}"
LOGIT = "K1 logit, entry congestion"  # option (a): the logit's linear request-year term extrapolates
print(f"v2 model: {CHOSEN}; refit matches notebook 11 (max gap {refit_gap:.1e}); {(time.time() - START) / 60:.1f} min")

## 2. What drifted

**2a. The level by entry year.** Observed P(operation by 3 and 5 years) of each test entry year with a bootstrap band, against the mean prediction of the v1 baseline, the fair baseline, the v1 LightGBM, the v2 model and the K1 logit, for all test requests and for the ≥ 20 MW cohort.

In [ ]:
durations, events = test["duration_years"].to_numpy(), test["event"].to_numpy()
years, regions = test["q_year"].to_numpy(), test["region"].astype(str).to_numpy()
MODELS = {"baseline": "baseline", "fair_baseline": "fair_baseline", "v1 LightGBM": "v1 LightGBM", "v2": "chosen", LOGIT: LOGIT}
preds = {h: {label: pred_v2[pred_v2["horizon"] == h].set_index("project_key").loc[test["project_key"].astype(str), column].to_numpy()
             for label, column in MODELS.items()} for h in config.HORIZONS}
rows = []
for cohort, mask in [("all", np.ones(len(test), bool)), ("≥ 20 MW", test["large"].to_numpy())]:
    for h in LEVEL_HORIZONS:
        t = drift.level_by_group({m: p[mask] for m, p in preds[h].items()}, durations[mask], events[mask], years[mask], h,
                                 n_boot=N_BOOT, seed=SEED + h)
        t.insert(0, "horizon", h)
        t.insert(0, "cohort", cohort)
        rows.append(t.rename(columns={"group": "entry_year"}))
by_year = pd.concat(rows, ignore_index=True).sort_values(["cohort", "horizon", "entry_year"])
by_year.to_csv(paths.RESULTS / "drift_by_cohort.csv", index=False)
by_region = []
for h in LEVEL_HORIZONS:
    for region in TRACK_A:
        mask = regions == region
        t = drift.level_by_group({m: p[mask] for m, p in preds[h].items()}, durations[mask], events[mask], years[mask], h,
                                 n_boot=N_BOOT, seed=SEED + 50 + h)
        t.insert(0, "horizon", h)
        t.insert(0, "region", region)
        by_region.append(t.rename(columns={"group": "entry_year"}))
by_region = pd.concat(by_region, ignore_index=True).sort_values(["horizon", "region", "entry_year"])
by_region.to_csv(paths.RESULTS / "drift_by_cohort_region.csv", index=False)
by_year.round(3)

**Reading this.** *(filled after the run)*

**2b. Which hazard moved.** The yearly hazards of operation and of withdrawal in the first three years in the queue, by entry cohort 2012–2019 (project-years from `discrete_time.observed_years`, as the models see them). If the operation hazard fell while the withdrawal hazard held, the drift is in how fast projects get built; if withdrawals rose, in how many survive to be built.

In [ ]:
cohorts = track_a[track_a["q_year"].between(2012, 2019)]
rows_c = discrete_time.observed_years(cohorts)
rows_c["q_year"] = cohorts.loc[rows_c["project"], "q_year"].to_numpy()
early = rows_c[rows_c["year_in_queue"] <= 3]
haz = (early.groupby("q_year")["outcome"].agg(project_years="size", h_operation=lambda s: (s == 1).mean(),
                                               h_withdrawal=lambda s: (s == 2).mean()).reset_index())
haz["se_operation"] = np.sqrt(haz["h_operation"] * (1 - haz["h_operation"]) / haz["project_years"])
haz["se_withdrawal"] = np.sqrt(haz["h_withdrawal"] * (1 - haz["h_withdrawal"]) / haz["project_years"])
haz.to_csv(paths.RESULTS / "drift_hazards_by_cohort.csv", index=False)

fig, axes = plt.subplots(1, 2)
for ax, cause, color in zip(axes, ["operation", "withdrawal"], [config.OUTCOME_COLORS["operation"], config.OUTCOME_COLORS["withdrawal"]]):
    h, se = haz[f"h_{cause}"], haz[f"se_{cause}"]
    ax.fill_between(haz["q_year"], (h - 1.96 * se).clip(lower=0), h + 1.96 * se, color=color, alpha=0.15, linewidth=0)
    ax.plot(haz["q_year"], h, color=color, marker="o")
    ax.axvspan(2016.5, 2019.5, color="#E5E5E5", zorder=0)
    ax.set_xlabel("Entry year")
    ax.set_title(f"Yearly hazard of {cause}, years 1–3 in the queue")
    ax.yaxis.set_major_formatter(lambda y, _: f"{y:.1%}")
axes[0].set_ylabel("Share of project-years ending in the event")
h_op = haz.set_index("q_year")["h_operation"]
h_wd = haz.set_index("q_year")["h_withdrawal"]
plots.finish(fig,
             f"In the first three years in the queue, the operation hazard fell from {h_op.loc[2016]:.1%} for 2016 entrants "
             f"to {h_op.loc[2018]:.1%} for 2018 entrants, while the withdrawal hazard went from {h_wd.loc[2016]:.1%} to {h_wd.loc[2018]:.1%}",
             "Share of project-years in the queue (years 1–3) that end in operation or withdrawal, by entry cohort; "
             "shaded band = ±1.96 standard errors; grey = the test cohorts",
             "Track A entrants 2012–2019, project-years under delayed entry")
save(fig, "12_01_hazards_by_cohort", "Early-year hazards of operation and withdrawal by entry cohort, 2012–2019.")
plt.show()
haz.round(4)

**Reading this.** *(filled after the run)*

## 3. Repairing the level

**Option (a), a trend term.** The discrete-time logit already carries the request year as a number, so its level extrapolates linearly; its line in table 2a shows what that buys.

**Option (b), rolling recalibration.** The v2 model keeps its ranking and gets its level from the newest resolved cohort before the one being scored: 2017 entrants from the 2016 cohort (predicted out of sample by the fit on entrants to 2015), 2018 from 2017, 2019 from 2018; an intercept per region and horizon, pooled when a region has fewer than 30 weighted source requests (`src/drift.rolling_recalibration`). The scored cohort's own outcomes never enter its intercept. The source cohort's outcomes have to be observed by the analysis date, so this is the level an analyst sets today from the newest cohort that has had time to resolve — not a level knowable on the request date; the report says so.

**Option (c), a Bayesian random walk on the cohort intercept** (PyMC), is a stretch item (decision #46) and is not fitted here.

In [ ]:
pool = pd.concat([cohort_2016, test])
pool_years = pool["q_year"].to_numpy()
pool_regions = pool["region"].astype(str).to_numpy()
pool_d, pool_e = pool["duration_years"].to_numpy(), pool["event"].to_numpy()
pool_follow = ((pd.Timestamp(config.CENSOR_DATE) - pool["q_date"]).dt.days / config.DAYS_PER_YEAR).to_numpy()
G_pool = evaluation.censoring_km(pool_follow, np.ones(len(pool), bool))
raw = {h: np.concatenate([p_2016[h].to_numpy(), p_test[h].to_numpy()]) for h in LEVEL_HORIZONS}
recal, intercepts = {}, []
for h in LEVEL_HORIZONS:
    y, w = evaluation.horizon_outcome(pool_d, pool_e, h, G_pool)
    out, table = drift.rolling_recalibration(raw[h], y, w, pool_years, pool_regions, SOURCE_YEAR)
    recal[h] = out[len(cohort_2016):]  # the test rows
    intercepts.append(table.assign(horizon=h))
intercepts = pd.concat(intercepts, ignore_index=True)
intercepts.to_csv(paths.RESULTS / "recalibration.csv", index=False)
print("mean predicted P(operation by 5 years), test cohort: raw "
      f"{preds[5]['v2'].mean():.1%} → recalibrated {np.nanmean(recal[5]):.1%}; observed {survival_tools.aalen_johansen(durations, events, [5])['cif_operation'].iloc[0]:.1%}")
intercepts.pivot_table(index=["horizon", "scored_year"], columns="region", values="intercept").round(2)

**Reading this.** *(filled after the run)*

## 4. The level on the 2018–2019 entrants (criterion C7) and figure F11

Calibration slope and intercept, with bootstrap intervals, of each option on the 2018–2019 entrants of 20 MW and more (C7's cohort), and for reference on all 2018–2019 entrants and on the 2017 entrants (whose source cohort, 2016, was predicted out of sample). An intercept shift leaves the ranking untouched, so Uno's C is unchanged by construction; it is reported once to show that.

In [ ]:
OPTIONS = {"v2 raw": lambda h: preds[h]["v2"], "v2 recalibrated": lambda h: recal[h], "K1 logit (trend)": lambda h: preds[h][LOGIT],
           "fair baseline": lambda h: preds[h]["fair_baseline"], "v1 baseline": lambda h: preds[h]["baseline"]}
COHORTS_C7 = {"2018–2019, ≥ 20 MW": (np.isin(years, [2018, 2019]) & test["large"].to_numpy()),
              "2018–2019, all": np.isin(years, [2018, 2019]),
              "2017, ≥ 20 MW": (years == 2017) & test["large"].to_numpy()}
follow_up = ((pd.Timestamp(config.CENSOR_DATE) - test["q_date"]).dt.days / config.DAYS_PER_YEAR).to_numpy()
rows = []
for c_i, (cohort, mask) in enumerate(COHORTS_C7.items()):
    G = evaluation.censoring_km(follow_up[mask], np.ones(int(mask.sum()), bool))
    for h in LEVEL_HORIZONS:
        sub = {name: f(h)[mask] for name, f in OPTIONS.items()}
        point = evaluation.score_horizon(sub, durations[mask], events[mask], h, G)
        draws = evaluation.bootstrap(lambda sw: evaluation.score_horizon(sub, durations[mask], events[mask], h, G, sw),
                                     int(mask.sum()), N_BOOT, SEED + 200 + 10 * c_i + h)
        t = evaluation.summarise(point, draws, reference="fair baseline", level=LEVEL)
        t.insert(0, "horizon", h)
        t.insert(0, "cohort", cohort)
        rows.append(t)
level_metrics = pd.concat(rows, ignore_index=True)
level_metrics.to_csv(paths.RESULTS / "level_repair_metrics.csv", index=False)
show = level_metrics[level_metrics["metric"].isin(["slope", "intercept", "uno_c"])].copy()
show["value"] = [f"{r.estimate:.2f} [{r.low:.2f}, {r.high:.2f}]" for r in show.itertuples()]
show.pivot_table(index=["cohort", "horizon", "metric"], columns="model", values="value", aggfunc="first")[list(OPTIONS)]

In [ ]:
fig, axes = plt.subplots(1, 2, sharey=True)
style = {"baseline": ("#999999", "s"), "fair_baseline": ("#555555", "D"), "v1 LightGBM": ("#CC79A7", "^"),
         "v2": ("#D55E00", "o"), "v2 recalibrated": ("#009E73", "o"), LOGIT: ("#0072B2", "v")}
labels = {"baseline": "v1 baseline", "fair_baseline": "Fair baseline", "v1 LightGBM": "v1 LightGBM", "v2": "v2 model, raw",
          "v2 recalibrated": "v2 model, recalibrated", LOGIT: "Logit (linear trend)"}
offsets = dict(zip(style, np.linspace(-0.3, 0.3, len(style))))
for ax, h in zip(axes, LEVEL_HORIZONS):
    part = by_year[(by_year["cohort"] == "≥ 20 MW") & (by_year["horizon"] == h)].set_index("entry_year").loc[TEST_YEARS]
    x = np.arange(len(TEST_YEARS))
    ax.errorbar(x, part["observed"], yerr=[part["observed"] - part["low"], part["high"] - part["observed"]], fmt="o",
                color=plots.INK, capsize=5, markersize=9, label="Observed, 95% band", zorder=4)
    large = test["large"].to_numpy()
    for m, (color, marker) in style.items():
        if m == "v2 recalibrated":
            values = [np.nanmean(recal[h][(years == y) & large]) for y in TEST_YEARS]
        else:
            values = part[m].to_numpy()
        ax.scatter(x + offsets[m], values, marker=marker, color=color, s=60, zorder=3, label=labels[m])
    ax.set_xticks(x, [f"{y}\n(n = {n:,})" for y, n in zip(TEST_YEARS, part["n"])])
    ax.set_title(f"P(operation by {h} years), requests ≥ 20 MW", fontsize=13)
    ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
axes[0].set_ylabel("Mean predicted / observed")
axes[1].legend(loc="upper right", fontsize=9)
lm = level_metrics.set_index(["cohort", "horizon", "model", "metric"])["estimate"]
plots.finish(fig,
             f"Recalibrating on the cohort before brings the v2 model's 5-year level for 2018–2019 entrants of 20 MW and more "
             f"from {lm[('2018–2019, ≥ 20 MW', 5, 'v2 raw', 'intercept')]:+.2f} to "
             f"{lm[('2018–2019, ≥ 20 MW', 5, 'v2 recalibrated', 'intercept')]:+.2f} on the intercept scale "
             f"(slope {lm[('2018–2019, ≥ 20 MW', 5, 'v2 recalibrated', 'slope')]:.2f})",
             "Observed P(operation by 3 and 5 years) per test entry year with 95% bootstrap bands, against each model's mean "
             "prediction; the recalibrated v2 model takes its level from the previous cohort",
             "Track A test entrants 2017–2019 of 20 MW and more")
plots.save_fig(fig, "F11_level_by_entry_year",
               "Observed against predicted P(operation by 3 and 5 years) by test entry year for requests of 20 MW and more, "
               "before and after the rolling recalibration.", NOTEBOOK)
plt.show()

**Reading this.** *(filled after the run)*

## 5. The newest evidence: 2020–2022 entrants

The v2 model (fitted on entrants to 2016) predicts P(operation by 1 and 2 years) for the 2020–2022 entrants of the three ISOs; notebook 01b measured what those cohorts actually did, in both editions of the file. The gap is the drift the models would carry into a 2026 decision, and the intercept that would close it at 2 years is reported as a diagnostic (not applied: the cohorts' 5-year level is not observable yet).

In [ ]:
recent = track_a[track_a["q_year"].between(2020, 2022)].copy()
p_recent = discrete_time.predict_both(fit_final, recent, horizons=[1, 2])[SPEC["model"]]
observed = pd.read_csv(paths.RESULTS / "early_incidence_2020_2022.csv")
rows = []
for year in [2020, 2021, 2022]:
    mask = (recent["q_year"] == year).to_numpy()
    for h in [1, 2]:
        for edition in [2025, 2026]:
            o = observed[(observed["entry_year"] == year) & (observed["horizon"] == h) & (observed["edition"] == edition)
                         & (observed["region"] == "CAISO+ERCOT+PJM")].iloc[0]
            rows.append({"entry_year": year, "horizon": h, "edition": edition, "n_predicted": int(mask.sum()),
                         "predicted": float(p_recent[h].to_numpy()[mask].mean()), "n_observed": int(o["n"]),
                         "observed": o["observed"], "low": o["low"], "high": o["high"]})
early_check = pd.DataFrame(rows)
# the intercept that would close the 2-year gap, from the 2025 edition's own outcomes (diagnostic only)
G_recent = evaluation.censoring_km(((pd.Timestamp(config.CENSOR_DATE) - recent["q_date"]).dt.days / config.DAYS_PER_YEAR).to_numpy(),
                                   np.ones(len(recent), bool))
y2, w2 = evaluation.horizon_outcome(recent["duration_years"].to_numpy(), recent["event"].to_numpy(), 2, G_recent)
a_needed = drift.fit_intercept(p_recent[2].to_numpy(), y2, w2)
early_check["intercept_needed_2yr_2025_edition"] = a_needed
early_check.to_csv(paths.RESULTS / "early_hazards_2020_2022.csv", index=False)

fig, axes = plt.subplots(1, 2, sharey=True)
for ax, h in zip(axes, [1, 2]):
    part = early_check[(early_check["horizon"] == h) & (early_check["edition"] == 2026)]
    x = np.arange(len(part))
    ax.errorbar(x - 0.12, part["observed"], yerr=[part["observed"] - part["low"], part["high"] - part["observed"]], fmt="o",
                color=plots.INK, capsize=5, markersize=8, label="Observed (2026 edition), 95% band")
    ax.scatter(x + 0.12, part["predicted"], marker="o", color="#D55E00", s=60, label="v2 model, fitted on entrants to 2016")
    ax.set_xticks(x, [f"{y}\n(n = {n:,})" for y, n in zip(part["entry_year"], part["n_predicted"])])
    ax.set_title(f"P(operation within {h} year{'s' if h > 1 else ''})", fontsize=13)
    ax.yaxis.set_major_formatter(lambda y, _: f"{y:.1%}")
axes[0].set_ylabel("Predicted / observed")
axes[1].legend(loc="upper right", fontsize=9)
ec = early_check[(early_check["horizon"] == 2) & (early_check["edition"] == 2026)].set_index("entry_year")
plots.finish(fig,
             f"For 2020–2022 entrants the v2 model predicts {ec.loc[2020, 'predicted']:.1%}, {ec.loc[2021, 'predicted']:.1%} and "
             f"{ec.loc[2022, 'predicted']:.1%} operation within two years; {ec.loc[2020, 'observed']:.1%}, "
             f"{ec.loc[2021, 'observed']:.1%} and {ec.loc[2022, 'observed']:.1%} did",
             "Mean predicted P(operation by 1 and 2 years) of the v2 model against the Aalen–Johansen incidence in the 2026 "
             "edition (data through 2025), CAISO, ERCOT and PJM entrants by entry year; 95% bootstrap bands",
             "2020–2022 entrants with event dates", lbnl_2026=True)
save(fig, "12_03_early_hazards_2020_2022", "Predicted against observed one- and two-year incidence of operation for 2020–2022 entrants.")
plt.show()
print(f"intercept that would close the 2-year gap (2025 edition outcomes): {a_needed:+.2f}")
early_check.round(4)

**Reading this.** *(filled after the run)*

## 6. Predictions for the later notebooks and checks

In [ ]:
out = []
for h in config.HORIZONS:
    f = pd.DataFrame({"project_key": test["project_key"].astype(str).to_numpy(), "horizon": h,
                      "v2_raw": preds[h]["v2"], "v2_recalibrated": recal[h] if h in recal else np.nan,
                      "fair_baseline": preds[h]["fair_baseline"], "baseline": preds[h]["baseline"],
                      "region": regions, "q_year": years, "large": test["large"].to_numpy(),
                      "event": events, "duration_years": durations})
    out.append(f)
pred_level = pd.concat(out, ignore_index=True)
pred_level.to_parquet(paths.DATA_PROCESSED / "predictions_v2_recalibrated.parquet", index=False)

checks = pd.DataFrame([
    {"check": "no scored cohort is its own level source", "value": str(SOURCE_YEAR),
     "pass": all(s < y for y, s in SOURCE_YEAR.items()) and (intercepts["source_year"] < intercepts["scored_year"]).all()},
    {"check": "the refit reproduces notebook 11's predictions", "value": f"{refit_gap:.1e}", "pass": refit_gap < 1e-6},
    {"check": "recalibration leaves the ranking unchanged (Uno's C equal)",
     "value": float(np.abs(lm.xs(("v2 raw", "uno_c"), level=("model", "metric")) - lm.xs(("v2 recalibrated", "uno_c"), level=("model", "metric"))).max()),
     "pass": bool(np.abs(lm.xs(("v2 raw", "uno_c"), level=("model", "metric")) - lm.xs(("v2 recalibrated", "uno_c"), level=("model", "metric"))).max() < 1e-9)},
    {"check": "every level metric carries an interval", "value": int(level_metrics[["low", "high"]].isna().any(axis=1).sum()),
     "pass": not level_metrics[["low", "high"]].isna().any(axis=1).any()},
    {"check": "F11 saved as PNG and SVG", "value": 2,
     "pass": (paths.FIGURES / "F11_level_by_entry_year.png").exists() and (paths.FIGURES / "F11_level_by_entry_year.svg").exists()},
])
checks.to_csv(paths.RESULTS / "level_checks.csv", index=False)
assert checks["pass"].all(), checks
print(f"total {(time.time() - START) / 60:.1f} min")
checks

## Takeaways

*(filled after the run)*